# Common topics
- The Aim of this notebook is to create topic clusters starting from different models to determine the common topics.
- The similarity metric used in this notebook is topic closeness.
- See the original paper at https://arxiv.org/abs/2412.18376

## Loading libraries

In [82]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [83]:
import pipeline.src.python.config as cfg
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
import networkx as nx
import leidenalg
import igraph as ig

## Loading models

In [4]:
MAGAZINE_1 = 'scopus'
DATASET_TEXT_FEATURE = (
    "text"  # In the dataset file, the column name that contains the text data
)
cfg_dict_1 = cfg.MAGAZINE_CONFIG[MAGAZINE_1]

In [9]:
MAGAZINE_2 = 'the_guardian'
cfg_dict_2 = cfg.MAGAZINE_CONFIG[MAGAZINE_2]

In [10]:
MAGAZINE_3 = 'science_news'
cfg_dict_3 = cfg.MAGAZINE_CONFIG[MAGAZINE_3]

In [11]:
from bertopic import BERTopic

model_path = cfg.MODELS_FOLDER / f'{MAGAZINE_1}/model_0.343.safetensors'
model_1 = BERTopic.load(model_path, 
                      embedding_model=cfg.EMBEDDING_MODEL
                      )

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
model_path_2 = cfg.MODELS_FOLDER / f'{MAGAZINE_2}/model_0.334.safetensors'

model_2 = BERTopic.load(model_path_2,
                        cfg.EMBEDDING_MODEL
                        )


In [13]:
model_path_3 = cfg.MODELS_FOLDER / f'{MAGAZINE_3}/model_0.309.safetensors'

model_3 = BERTopic.load(model_path_3,
                        cfg.EMBEDDING_MODEL
                        )


## Loading data

In [15]:
data = np.load(cfg_dict_1['OUTPUT_PATH'],allow_pickle=True) 

ids = data['id']
texts = data['text'] 
embeddings = data['embedding'] 
documents = data['clean_text']

In [ ]:
data_2 = np.load(cfg_dict_2['OUTPUT_PATH'],allow_pickle=True) 

ids_2 = data_2['id']
texts_2 = data_2['text'] 
embeddings_2 = data_2['embedding'] 
documents_2 = data_2['clean_text']

In [17]:
data_3 = np.load(cfg_dict_3['OUTPUT_PATH'],allow_pickle=True) 

ids_3 = data_3['id']
texts_3 = data_3['text'] 
embeddings_3 = data_3['embedding'] 
documents_3 = data_3['clean_text']

## BTM Metrics evaluation - Model 1 vs Model 2

### First side evalutation - Model 1 -> Model 2

In [ ]:
from pipeline.src.python.btm import BTM

models_metrics_1_vs_2 = BTM(model_1=model_1,
                            model_2=model_2,
                            ids_1=ids,
                            texts_1=texts,
                            embeddings_1=embeddings,
                            texts_2=texts_2,
                            embeddings_2=embeddings_2,
                            model_1_name=MAGAZINE_1.title(),
                            model_2_name=MAGAZINE_2.title())

2026-02-09 14:09:37,164 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:37,370 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3820500373840332


In [25]:
models_metrics_1_vs_2.evaluate_metrics()

In [26]:
model_1_vs_model_2_closeness = models_metrics_1_vs_2.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

### Second side evaluation - Model 2 -> Model 1

In [ ]:
inverse_models_metrics_2_vs_1 = BTM(model_1=model_2,
                                    model_2=model_1,
                                    ids_1=ids_2,
                                    texts_1=texts_2,
                                    embeddings_1=embeddings_2,
                                    texts_2=texts,
                                    embeddings_2=embeddings,
                                    model_1_name=MAGAZINE_2.title(),model_2_name=MAGAZINE_1.title())

2026-02-09 14:09:40,871 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:42,092 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.38246655464172363


In [28]:
inverse_models_metrics_2_vs_1.evaluate_metrics()

In [29]:
model_2_vs_model_1_closeness = inverse_models_metrics_2_vs_1.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

## BTM Metrics evaluation - Model 1 vs Model 3

### First side evaluation - Model 1 -> Model 3

In [ ]:
from pipeline.src.python.btm import BTM

models_metrics_1_vs_3 = BTM(model_1=model_1,
                            model_2=model_3,
                            ids_1=ids,
                            texts_1=texts,
                            embeddings_1=embeddings,
                            texts_2=texts_3,
                            embeddings_2=embeddings_3,
                            model_1_name=MAGAZINE_1.title(),
                            model_2_name=MAGAZINE_3.title())

2026-02-09 14:09:54,709 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:54,721 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3790895342826843


In [31]:
models_metrics_1_vs_3.evaluate_metrics()

In [32]:
model_1_vs_model_3_closeness = models_metrics_1_vs_3.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

### Second side evaluation - Model 3 -> Model 1

In [ ]:
inverse_models_metrics_3_vs_1 = BTM(model_1=model_3,
                                    model_2=model_1,
                                    ids_1=ids_3,
                                    texts_1=texts_3,
                                    embeddings_1=embeddings_3,
                                    texts_2=texts,
                                    embeddings_2=embeddings,
                                    model_1_name=MAGAZINE_3.title(),
                                    model_2_name=MAGAZINE_1.title())

2026-02-09 14:11:39,363 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:11:40,577 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.38246655464172363


In [34]:
inverse_models_metrics_3_vs_1.evaluate_metrics()

In [35]:
model_3_vs_model_1_closeness = inverse_models_metrics_3_vs_1.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

## BTM Metrics evaluation - Model 2 vs Model 3

### First side evaluation - Model 2 -> Model 3

In [36]:
from pipeline.src.python.btm import BTM

models_metrics_2_vs_3 = BTM(model_1=model_2,
                            model_2=model_3,
                            ids_1=ids_2,
                            texts_1=texts_2,
                            embeddings_1=embeddings_2,
                            texts_2=texts_3,
                            embeddings_2=embeddings_3,
                            model_1_name=MAGAZINE_2.title(),
                            model_2_name=MAGAZINE_3.title()
                            )

2026-02-09 14:14:37,245 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:14:37,257 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3790895342826843


In [37]:
models_metrics_2_vs_3.evaluate_metrics()

In [38]:
model_2_vs_model_3_closeness = models_metrics_2_vs_3.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

### Second side evaluation - Model 3 -> Model 2

In [39]:
inverse_models_metrics_3_vs_2 = BTM(model_1=model_3,
                                    model_2=model_2,
                                    ids_1=ids_3,
                                    texts_1=texts_3,
                                    embeddings_1=embeddings_3,
                                    texts_2=texts_2,
                                    embeddings_2=embeddings_2,
                                    model_1_name=MAGAZINE_3.title(),
                                    model_2_name=MAGAZINE_2.title())

2026-02-09 14:16:10,197 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:16:10,400 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3820500373840332


In [41]:
inverse_models_metrics_3_vs_2.evaluate_metrics()

In [40]:
model_3_vs_model_2_closeness = inverse_models_metrics_3_vs_2.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

## Saving data to parquet

### Standardize column names

In [ ]:
model_1_vs_model_2_closeness = model_1_vs_model_2_closeness.rename(columns={f'{MAGAZINE_1.title()} Topic Label':'Model 1 Label',
                                                                            f'{MAGAZINE_2.title()} Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [ ]:
model_2_vs_model_1_closeness = model_2_vs_model_1_closeness.rename(columns={f'{MAGAZINE_1.title()} Topic Label':'Model 2 Label',
                                                                            f'{MAGAZINE_2.title()} Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [ ]:
model_1_vs_model_3_closeness = model_1_vs_model_3_closeness.rename(columns={f'{MAGAZINE_1.title()} Topic Label':'Model 1 Label',
                                                                            f'{MAGAZINE_3.title()} Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [ ]:
model_3_vs_model_1_closeness = model_3_vs_model_1_closeness.rename(columns={f'{MAGAZINE_1.title()} Topic Label':'Model 2 Label',
                                                                            f'{MAGAZINE_3.title()} Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [ ]:
model_2_vs_model_3_closeness = model_2_vs_model_3_closeness.rename(columns={f'{MAGAZINE_2.title()} Topic Label':'Model 1 Label',
                                                                            f'{MAGAZINE_3.title()} Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [ ]:
model_3_vs_model_2_closeness = model_3_vs_model_2_closeness.rename(columns={f'{MAGAZINE_2.title()} Topic Label':'Model 2 Label',
                                                                            f'{MAGAZINE_3.title()} Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

### Create and save edges dataset

In [58]:
closeness = pd.concat([model_1_vs_model_2_closeness,
                model_2_vs_model_1_closeness,
                model_1_vs_model_3_closeness,
                model_3_vs_model_1_closeness,
                model_2_vs_model_3_closeness,
                model_3_vs_model_2_closeness])


In [63]:
closeness.to_parquet('edges.parquet')

### Create and save nodes dataset

In [ ]:
counts = model_1.get_topic_freq()['Count'][1:].to_list() if -1 in model_1.topics_ else model_1.get_topic_freq()['Count'].to_list()
labels  = model_1.custom_labels_[1:] if -1 in model_1.topics_ else model_1.custom_labels_

model_1_nodes = pd.DataFrame({'Topic Label': labels,
                              'Degree':counts })

model_1_nodes['Model'] = MAGAZINE_1.title()


In [28]:
counts_2 = model_2.get_topic_freq()['Count'][1:].to_list() if -1 in model_2.topics_ else model_2.get_topic_freq()['Count'].to_list()
labels_2  = model_2.custom_labels_[1:] if -1 in model_2.topics_ else model_2.custom_labels_

model_2_nodes = pd.DataFrame({'Topic Label': labels_2,
                              'Degree':counts_2 })
model_2_nodes['Model'] = MAGAZINE_2.title()

In [30]:
counts_3 = model_3.get_topic_freq()['Count'][1:].to_list() if -1 in model_3.topics_ else model_3.get_topic_freq()['Count'].to_list()
labels_3  = model_3.custom_labels_[1:] if -1 in model_3.topics_ else model_3.custom_labels_

model_3_nodes = pd.DataFrame({'Topic Label': labels_3,
                              'Degree':counts_3 })
model_3_nodes['Model'] = MAGAZINE_3.title()

In [32]:
nodes = pd.concat([model_1_nodes,
                model_2_nodes,
                model_3_nodes])

In [33]:
nodes.to_parquet('nodes.parquet')

## Clustering Analysis

### Retriving data from parquet

In [84]:
import pandas as pd
edges = pd.read_parquet('edges.parquet')
nodes = pd.read_parquet('nodes.parquet')

### Edge filtering

- Filter all the edges that don't have bidirectional linkage
- Filter all the edges that have a Topic Closeness less than 0.1
- Filter all the edges with less than 5 match

In [85]:
enriched_edges = edges.merge(nodes,how='inner',left_on='Model 1 Label',right_on='Topic Label')

In [86]:
enriched_edges['N_Match'] = enriched_edges['Topic Closeness']  * enriched_edges['Degree']

In [87]:
enriched_edges = enriched_edges[ enriched_edges['N_Match'] > 5 ]

In [88]:
filtered_topics_1 = []
filtered_topics_2 = []
closeness = []

for _, edge in enriched_edges.iterrows():

    topic1 = edge['Model 1 Label']
    topic2 = edge['Model 2 Label']
    c = edge['Topic Closeness']

    if c <= 0.1:
        continue

    bidirection = edges[
        (edges['Model 1 Label'] == topic2) &
        (edges['Model 2 Label'] == topic1) &
        (edges['Topic Closeness'] > 0.1)
    ]

    if not bidirection.empty:

        filtered_topics_1.append(topic1)
        filtered_topics_2.append(topic2)
        closeness.append(c)

In [89]:
filtered_edges = pd.DataFrame({'Model 1 Label': filtered_topics_1,
                        'Model 2 Label': filtered_topics_2,
                        'Topic Closeness':closeness})

### Graph creation

In [90]:
G = nx.DiGraph(description='Connected components') 

for _, node in nodes.iterrows():
    G.add_node(
    node['Topic Label'],
    degree=node['Degree'],
    model=node['Model']
)

for _, edge in filtered_edges.iterrows():
    G.add_edge(
    edge['Model 1 Label'],
    edge['Model 2 Label'],
    weight=edge['Topic Closeness']
)

### Similarity operation

We need to transform directed graph into undirected graph.
To do it, we need to decide how combine the weights of the two edges.

The options are:
- Min
- Harmonic mean

To be more conservative, we adopt the min between two weights.

In [91]:
dir_weights = {}

for u, v, data in G.edges(data=True):
    w = data["weight"]
    dir_weights[(u, v)] = w

In [92]:
def harmonic_mean(a, b, eps=1e-9):
    return 2*a*b/(a+b+eps)

In [93]:
G_und = nx.Graph()

for (u, v), w_uv in dir_weights.items():

    if (v, u) in dir_weights:
        w_vu = dir_weights[(v, u)]

        w = min(w_uv, w_vu)
        #w = harmonic_mean(w_uv, w_vu)

        if not G_und.has_edge(u, v):
            G_und.add_edge(u, v, weight=w)


### Community algorithm execution

We adopt the Leiden algorithm to build a primary community structure.

See more about it on https://medium.com/@swapnil.agashe456/leiden-clustering-for-community-detection-a-step-by-step-guide-with-python-implementation-c883933a1430

In [94]:
nodes = list(G_und.nodes())
G_ig = ig.Graph.from_networkx(G_und)
G_ig.vs["name"] = nodes

In [95]:
partition = leidenalg.find_partition(
    G_ig,
    leidenalg.ModularityVertexPartition,
    weights="weight",
    n_iterations=-1
)

In [96]:
clusters = partition.membership

In [97]:
node_to_cluster = {
    v["name"]: clusters[i]
    for i, v in enumerate(G_ig.vs)
}

In [98]:
from collections import defaultdict

clusters = defaultdict(list)

for node, comm in node_to_cluster.items():
    clusters[comm].append(node)

for c, nodes in clusters.items():
    print(f"Cluster {c}:")
    print(", ".join(nodes))
    print()


Cluster 24:
Tuberculosis Resistance and Control, Tuberculosis Treatment Challenges

Cluster 0:
Zoonotic Disease Surveillance, Disease Spread and Environmental Impact, WNV Surveillance and Detection, Zika Outbreak and Mosquito Spread, Zika Virus Outbreaks, Zika Virus Outbreak Risk, Zika Virus and Microcephaly, Climate and Health Impact, Disease Outbreak Tracking Initiative

Cluster 8:
Infection Control and Prevention, Hospital Infection Control Improvement, Hospital Surface Sterilization and Contamination Control, MRSA Hospital Surveillance

Cluster 1:
SARS-CoV-2 Genomic Evolution, Coronavirus Death Trends in England, Covid-19 Pandemic Mitigation Strategies, COVID-19 Spread and Mortality, Global Coronavirus Outbreak Death Toll, School Outbreak Transmission, School Reopening and Educational Needs, Coronavirus Death Toll Surpasses Million

Cluster 4:
HPAI Outbreak and Pathogenicity, H5N1 Bird Flu Outbreak in Poultry Industry, Flu Virus Spread Patterns, Viral Replication in Influenza Virus

In [99]:
label_name = []
cluster_number = []
for k,v in node_to_cluster.items():
    label_name.append(k)
    cluster_number.append(v)


In [100]:
print(f"Number of communities: {sorted(cluster_number)[-1]+1}")

Number of communities: 45


In [101]:
leiden_cluster = pd.DataFrame({'Topic Label':label_name,
              'Cluster': cluster_number})

### Enrichment phase

Now, we enrich the primary community structure using the similarity between excluded topics and cluster members.

In [ ]:
# The threshold that permit to decide when stop the insertion of unclustered topics
MEAN_CLOSENESS_THRESHOLD = 0.1

i = 0
leiden_cluster.sort_values(by='Cluster').to_csv(f'primary_community_structure.csv',sep='ç',encoding='utf-8')
print('Primary community structure saved')

In [ ]:
## Enrichment loop

while True:
    # Add the cluster size to the dataframe
    leiden_cluster = leiden_cluster.merge(leiden_cluster['Cluster'].value_counts().reset_index().rename(columns={'count':'Cluster size'}),on='Cluster')

    # Consider only topics without cluster
    unclustered_topics = edges[ ~ edges['Model 1 Label'].isin(leiden_cluster['Topic Label'])]

    # Consider only topics with at least one edge with clustered topics
    unclustered_topics_with_closeness = unclustered_topics.merge(leiden_cluster,how='left',left_on='Model 2 Label',right_on='Topic Label')
    unclustered_topics_with_closeness = unclustered_topics_with_closeness[ ~ unclustered_topics_with_closeness['Topic Label'].isna()]

    # For each unclustered topic, sum the best 3 edges with the cluster members
    k = 3
    top_k_closeness = ( 
    unclustered_topics_with_closeness
    .groupby(['Model 1 Label','Cluster','Cluster size'],as_index=False)
    .head(k) 
    .groupby(['Model 1 Label','Cluster','Cluster size'])['Topic Closeness']
    .agg(Topic_Closeness_Sum="sum")
    )

    # Compute the mean by k (even if there aren't enough connections)
    top_k_closeness = top_k_closeness.reset_index()
    top_k_closeness['Topic Closeness Mean'] = top_k_closeness['Topic_Closeness_Sum'] / k

    unclustered_topics_with_closeness_aggregated = top_k_closeness.copy()

    # Aggregate and select the best cluster that has the best mean score
    the_best_unclustered_topics = unclustered_topics_with_closeness_aggregated.loc[ 
    unclustered_topics_with_closeness_aggregated.groupby(['Model 1 Label'])["Topic Closeness Mean"].idxmax()
    ]
    
    the_best_unclustered_topics = the_best_unclustered_topics.astype({'Cluster':"int64","Cluster size":"int64"})

    # Select the best 10 uncluster topic and add to appropriate clusters
    added_topics = the_best_unclustered_topics[ the_best_unclustered_topics['Topic Closeness Mean'] >= MEAN_CLOSENESS_THRESHOLD ].sort_values(by='Topic Closeness Mean',ascending=False)[['Model 1 Label','Cluster','Cluster size']]
    added_topics = added_topics.rename(columns={'Model 1 Label':'Topic Label'})
    leiden_cluster = pd.concat([leiden_cluster,added_topics.head(10)])

    leiden_cluster = leiden_cluster.drop(columns=['Cluster size'])

    # Save the run outcome
    if len(added_topics) > 0:
        i += 1
        print(f'Run {i}: {len(added_topics.head(10))} added')
        leiden_cluster.sort_values(by='Cluster').to_csv(f'community_enrichment_run_{i}.csv',sep='ç',encoding='utf-8')
    else:
        print(f'All the unclustered topics below the {MEAN_CLOSENESS_THRESHOLD} threshold were assigned to the clusters')
        break

Run 1: 104 added
Run 2: 92 added
Run 3: 81 added
Run 4: 70 added
Run 5: 59 added
Run 6: 50 added
Run 7: 41 added
Run 8: 31 added
Run 9: 21 added
Run 10: 11 added
Run 11: 2 added
Run 12: 3 added
All the unclustered topics below the 0.1 has been assigned to the clusters
